# 04 — Plot Slopes on Surface Brain

Map parcel slopes back to fsaverage6 vertices and render using `nilearn.plotting.plot_surf_stat_map`.
Each figure shows 4 views: LH lateral, LH medial, RH lateral, RH medial.

Produces:
- Per-subject plots: `figures/brain_maps/individual/{subject}_{task}_{contrast}.png`
- Group-average plots: `figures/brain_maps/group/{task}_{contrast}.png`

In [ ]:
import sys
import pickle
import tempfile
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from nilearn import plotting, datasets

try:
    _nb_dir = Path(__file__).resolve().parent
except NameError:
    _nb_dir = Path.cwd()
sys.path.insert(0, str(_nb_dir))
from config import (
    TASKS, CONTRASTS, SUBJECTS, RT_CONTRAST,
    N_VERTICES_PER_HEMI, DATA_DIR, FIGURES_DIR,
    get_schaefer_surface_labels,
)

with open(DATA_DIR / 'surface_indiv_slopes.pkl', 'rb') as f:
    indiv_slopes = pickle.load(f)
with open(DATA_DIR / 'surface_avg_slopes.pkl', 'rb') as f:
    avg_slopes = pickle.load(f)

labels, parcel_names = get_schaefer_surface_labels()
fsavg6 = datasets.fetch_surf_fsaverage('fsaverage6')

FIG_DIR = FIGURES_DIR / 'brain_maps'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Loaded.')

In [ ]:
def slopes_to_vertices(slope_array, labels, n_vertices=81924):
    """
    Map (400,) parcel slope array back to (81924,) per-vertex array.
    Medial wall vertices (label=401) are set to 0.
    """
    vertex_slopes = np.zeros(n_vertices)
    for i, slope in enumerate(slope_array):
        vertex_slopes[labels == (i + 1)] = slope
    return vertex_slopes


def plot_four_views(lh_data, rh_data, vmax, title, save_path):
    """
    Render LH lateral/medial + RH lateral/medial as a 2×2 grid.
    Saves a single PNG at save_path.
    """
    views = [
        (fsavg6.pial_left,  lh_data, fsavg6.sulc_left,  'lateral', 'LH lateral'),
        (fsavg6.pial_left,  lh_data, fsavg6.sulc_left,  'medial',  'LH medial'),
        (fsavg6.pial_right, rh_data, fsavg6.sulc_right, 'lateral', 'RH lateral'),
        (fsavg6.pial_right, rh_data, fsavg6.sulc_right, 'medial',  'RH medial'),
    ]

    tmpfiles = []
    for mesh, data, bg, view, label in views:
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
        tmp.close()
        plotting.plot_surf_stat_map(
            mesh, data, bg_map=bg, view=view,
            cmap='RdBu_r', vmax=vmax, colorbar=True,
            title=label, output_file=tmp.name,
        )
        plt.close('all')
        tmpfiles.append(tmp.name)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, tmpf in zip(axes.flat, tmpfiles):
        ax.imshow(mpimg.imread(tmpf))
        ax.axis('off')
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.close()

    for f in tmpfiles:
        os.unlink(f)

In [ ]:
# Per-subject brain maps
indiv_dir = FIG_DIR / 'individual'
indiv_dir.mkdir(parents=True, exist_ok=True)

for subject in SUBJECTS:
    for task in TASKS:
        for contrast in CONTRASTS[task]:
            if contrast == RT_CONTRAST:
                continue
            data = indiv_slopes.get(subject, {}).get(task, {}).get(contrast, {})
            if not data:
                continue

            slope_arr = np.array([data.get(p, {}).get('beta_slope', 0.0) for p in parcel_names])
            vertex_slopes = slopes_to_vertices(slope_arr, labels)

            # Symmetric colorscale at 98th percentile
            vmax = np.percentile(np.abs(vertex_slopes[vertex_slopes != 0]), 98) if (vertex_slopes != 0).any() else 1.0

            plot_four_views(
                lh_data   = vertex_slopes[:N_VERTICES_PER_HEMI],
                rh_data   = vertex_slopes[N_VERTICES_PER_HEMI:],
                vmax      = vmax,
                title     = f'{subject} | {task} | {contrast}',
                save_path = indiv_dir / f'{subject}_{task}_{contrast}.png',
            )
    print(f'  Done: {subject}')

print(f'Individual brain maps saved to {indiv_dir}')

In [ ]:
# Group-average brain maps
group_dir = FIG_DIR / 'group'
group_dir.mkdir(parents=True, exist_ok=True)

# Global 98th-percentile vmax across all contrasts for consistent scaling
all_group_slopes = []
for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        data = avg_slopes.get(task, {}).get(contrast, {})
        if data:
            all_group_slopes.extend([data[p]['slope_mean'] for p in parcel_names if p in data])
global_vmax = np.percentile(np.abs(all_group_slopes), 98) if all_group_slopes else 1.0

for task in TASKS:
    for contrast in CONTRASTS[task]:
        if contrast == RT_CONTRAST:
            continue
        data = avg_slopes.get(task, {}).get(contrast, {})
        if not data:
            continue

        slope_arr = np.array([data.get(p, {}).get('slope_mean', 0.0) for p in parcel_names])
        vertex_slopes = slopes_to_vertices(slope_arr, labels)

        plot_four_views(
            lh_data   = vertex_slopes[:N_VERTICES_PER_HEMI],
            rh_data   = vertex_slopes[N_VERTICES_PER_HEMI:],
            vmax      = global_vmax,
            title     = f'Group mean | {task} | {contrast}',
            save_path = group_dir / f'{task}_{contrast}.png',
        )
    print(f'  Done: {task}')

print(f'Group brain maps saved to {group_dir}')